 ÉTAPE 4 — API FLASK  
 

In [1]:
import sys
import os

# ── Redémarrage propre : si vous voyez une RecursionError plus bas,
#    Kernel > Restart Kernel, puis réexécutez TOUT depuis cette cellule.
sys.setrecursionlimit(5000)

from flask import Flask, jsonify, render_template, request
import numpy as np
import pandas as pd
import joblib
import csv
from datetime import datetime
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import io, base64

# ── app créée UNE SEULE FOIS, ici, jamais recréée ailleurs ─────────────
app = Flask(__name__)

# ── Config (déplacée ici pour être définie avant toute utilisation) ────
HISTORIQUE_CSV = "historique_predictions.csv"
SEUIL_CRITIQUE = 0.20
SEUIL_FAIBLE   = 0.50


4.1  Initialisation ─────────────────────────────────────────────────────

In [2]:

# ======================
# SAFE LOAD MODEL
# ======================
try:
    model = joblib.load("model.pkl")
    print("model chargé OK -", type(model).__name__)
except Exception as e:
    print("ERREUR model.pkl :", e)
    model = None


# ======================
# SAFE LOAD SCALER
# ======================
try:
    scaler = joblib.load("scaler.pkl")
    print("scaler chargé OK")
except Exception as e:
    print("ERREUR scaler.pkl :", e)
    scaler = None


# ======================
# METRICS SAFE
# ======================
try:
    metrics_raw = joblib.load("metriques.pkl")

    metrics = {
        "mae": round(float(metrics_raw["mae"]), 2),
        "rmse": round(float(metrics_raw["rmse"]), 2),
        "r2": round(float(metrics_raw["r2"]), 4),
    }

except Exception as e:
    print("ERREUR metriques.pkl :", e)
    metrics = {"mae": "N/A", "rmse": "N/A", "r2": "N/A"}


model chargé OK - RandomForestRegressor
scaler chargé OK


4.2  Fonctions utilitaires ──────────────────────────────────────────────

In [4]:
def predire_ventes(stock, purchase, annee, mois, jour, jour_semaine):

    if model is None:
        raise Exception("❌ Model non chargé (model.pkl est NULL)")

    data = np.array([[stock, purchase, annee, mois, jour, jour_semaine]])

    # ── IMPORTANT : le modèle (RandomForest) a été entraîné SANS scaler
    #    dans Preparation_Et_EntrainementML.ipynb / SYSTEME_DE_RECOMMANDATION.
    #    On ne transforme donc PAS les données ici, pour rester cohérent.
    #    Si un jour vous entraînez avec scaler.fit_transform(), décommentez :
    # if scaler is not None:
    #     data = scaler.transform(data)

    return round(float(model.predict(data)[0]), 2)


# ── RECOMMANDATION ───────────────────────────────────
def recommandation(stock, ventes_prevues):
    ratio = stock / ventes_prevues if ventes_prevues > 0 else 1
    quantite = max(0, round(ventes_prevues - stock))

    if ratio < SEUIL_CRITIQUE:
        niveau = "CRITIQUE"
        decision = f"Commander {quantite} unités IMMÉDIATEMENT"
    elif ratio < SEUIL_FAIBLE:
        niveau = "FAIBLE"
        decision = f"Passer commande de {quantite} unités bientôt"
    elif stock < ventes_prevues:
        niveau = "ATTENTION"
        decision = f"Stock insuffisant ({quantite} manquantes)"
    else:
        niveau = "SUFFISANT"
        decision = "Stock suffisant"

    return {
        "ventes_prevues": ventes_prevues,
        "quantite_a_commander": quantite,
        "ratio_stock": round(ratio * 100, 1),
        "niveau": niveau,
        "decision": decision
    }

# ── HISTORIQUE ───────────────────────────────────────
def sauvegarder_historique(stock, purchase, annee, mois, jour, jour_semaine, ventes, result):

    existe = os.path.exists(HISTORIQUE_CSV)

    with open(HISTORIQUE_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not existe:
            writer.writerow([
                "date", "stock", "purchase", "annee", "mois",
                "jour", "jour_semaine", "ventes", "niveau", "quantite"
            ])

        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M"),
            stock, purchase, annee, mois, jour, jour_semaine,
            ventes, result["niveau"], result["quantite_a_commander"]
        ])

# ── GRAPH ─────────────────────────────────────────────
def generer_graphique_mini(stock, ventes):

    fig, ax = plt.subplots(figsize=(4, 2.5))
    bars = ax.bar(["Stock", "Ventes"], [stock, ventes],
                  color=["#1a73e8", "#35ea6e"])

    ax.bar_label(bars, fmt="%.0f")
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format="png")
    plt.close()

    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")


4.3  Routes Flask ───────────────────────────────────────────────────────

In [5]:
# ======================
# HOME
# ======================
@app.route("/")
def home():
    return render_template(
        "index.html",
        metrics=metrics,
        prediction=None
    )

# ======================
# PREDICTION
# ======================
@app.route("/predict", methods=["POST"])
def predict():

    try:
        stock = float(request.form["stock"])
        purchase = float(request.form["purchase"])

        date = pd.to_datetime(request.form["date"])

        annee = date.year
        mois = date.month
        jour = date.day
        jour_semaine = date.dayofweek

        ventes = predire_ventes(stock, purchase, annee, mois, jour, jour_semaine)

        print("DEBUG ventes =", ventes)

        result = recommandation(stock, ventes)

        sauvegarder_historique(
            stock, purchase, annee, mois, jour, jour_semaine, ventes, result
        )

        graphique = generer_graphique_mini(stock, ventes)

        return render_template(
            "index.html",
            prediction=result.get("ventes_prevues"),
            decision_text=result.get("decision"),
            niveau=result.get("niveau"),
            quantite=result.get("quantite_a_commander"),
            ratio_stock=result.get("ratio_stock"),
            graphique_base64=graphique,
            metrics=metrics
        )

    except Exception as e:
        print("ERROR:", e)
        return render_template(
            "index.html",
            error=str(e),
            metrics=metrics
        )


# ======================
# HISTORIQUE API
# ======================
@app.route("/historique")
def historique():

    if not os.path.exists(HISTORIQUE_CSV):
        return jsonify({"historique": []})

    try:
        df = pd.read_csv(HISTORIQUE_CSV)
        return jsonify({"historique": df.to_dict(orient="records")})
    except Exception as e:
        return jsonify({"historique": [], "error": str(e)})


# ======================
# METRICS API
# ======================
@app.route("/metriques")
def metriques():
    return jsonify(metrics)


 4.4  Lancement ──────────────────────────────────────────────────────────

In [ ]:
if __name__ == "__main__":
   app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [20/Jun/2026 17:00:17] "GET / HTTP/1.1" 200 -
c:\Users\disfra\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
127.0.0.1 - - [20/Jun/2026 17:00:37] "POST /predict HTTP/1.1" 200 -


DEBUG ventes = 22.69


127.0.0.1 - - [20/Jun/2026 17:01:58] "GET / HTTP/1.1" 200 -
c:\Users\disfra\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
127.0.0.1 - - [20/Jun/2026 17:02:10] "POST /predict HTTP/1.1" 200 -


DEBUG ventes = 28.36


127.0.0.1 - - [21/Jun/2026 19:17:11] "GET / HTTP/1.1" 200 -
c:\Users\disfra\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


DEBUG ventes = 28.72


127.0.0.1 - - [21/Jun/2026 19:17:30] "POST /predict HTTP/1.1" 200 -


 Objectif : exposer le modèle via une API web Flask. 
 Routes :
 GET  /           → page principale (index.html)
 POST /predict    → prédiction + recommandation
 GET  /historique → tableau des prédictions passées
 GET  /metriques  → précision du modèle (JSON)